#MNB

In [16]:
import time
import numpy as np
from scipy import sparse
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB   # ONLY used to check our work

In [17]:
class MyMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)          # sorted list of class labels
        n_words = X.shape[1]

        # N_ci: how many times word i appears in class c (one row per class)
        counts = np.vstack([
            np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_
        ])

        # Eq. 4: smoothed probability of word i in class c
        theta = (counts + self.alpha) / (
            counts.sum(axis=1, keepdims=True) + self.alpha * n_words)

        # Eq. 5: weights are log-probabilities
        self.w_ = np.log(theta)
        return self

    def decision_function(self, X):
        # Score of each class = sum over words of (count x weight)
        X = sparse.csr_matrix(X, dtype=np.float64)
        return np.asarray(X @ self.w_.T)

    def predict(self, X):
        # Highest score wins
        return self.classes_[self.decision_function(X).argmax(axis=1)]

In [18]:
# vocabulary order: goal, match, vote
X_toy = np.array([[2, 1, 0],    # sports:   "goal goal match"
                  [1, 2, 0],    # sports:   "goal match match"
                  [0, 1, 2]])   # politics: "vote vote match"
y_toy = np.array(["sports", "sports", "politics"])

toy = MyMultinomialNB(alpha=1.0).fit(X_toy, y_toy)
print("classes:", toy.classes_)
print("weights:\n", toy.w_.round(3))

X_new = np.array([[2, 0, 1]])   # "goal goal vote"
print("scores:", toy.decision_function(X_new).round(3))
print("prediction:", toy.predict(X_new))

classes: ['politics' 'sports']
weights:
 [[-1.792 -1.099 -0.693]
 [-0.811 -0.811 -2.197]]
scores: [[-4.277 -3.819]]
prediction: ['sports']


In [13]:
data = fetch_20newsgroups(subset="all")
train_text, test_text, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2,
    stratify=data.target, random_state=0)

vectorizer = CountVectorizer()                   # turns text into word counts
X_train = vectorizer.fit_transform(train_text)   # vocabulary learned from TRAIN only
X_test = vectorizer.transform(test_text)
print("train:", X_train.shape, "  test:", X_test.shape)

train: (15076, 152615)   test: (3770, 152615)


In [14]:
results = []

def evaluate(name, model):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start
    pred = model.predict(X_test)
    row = {"model": name,
           "accuracy": round(accuracy_score(y_test, pred), 4),
           "macro_f1": round(f1_score(y_test, pred, average="macro"), 4),
           "fit_time_s": round(fit_time, 3)}
    results.append(row)
    print(row)
    return pred

In [15]:
mnb = MyMultinomialNB(alpha=1.0)
pred_mnb = evaluate("MNB", mnb)

ref = MultinomialNB(alpha=1.0, fit_prior=False).fit(X_train, y_train)
print("Same predictions as scikit-learn:", np.array_equal(pred_mnb, ref.predict(X_test)))
print("Largest weight difference:", np.abs(mnb.w_ - ref.feature_log_prob_).max())

{'model': 'MNB', 'accuracy': 0.8621, 'macro_f1': 0.8465, 'fit_time_s': 0.072}
Same predictions as scikit-learn: True
Largest weight difference: 1.7763568394002505e-15


#CNB

In [19]:
class MyNB:
    def __init__(self, alpha=1.0, complement=False):
        self.alpha = alpha
        self.complement = complement

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_words = X.shape[1]

        counts = np.vstack([
            np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_
        ])

        # NEW (Step 2, Eq. 6): complement counts = all classes minus class c itself
        if self.complement:
            counts = counts.sum(axis=0, keepdims=True) - counts

        theta = (counts + self.alpha) / (
            counts.sum(axis=1, keepdims=True) + self.alpha * n_words)
        self.w_ = np.log(theta)
        return self

    def decision_function(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64)
        return np.asarray(X @ self.w_.T)

    def predict(self, X):
        scores = self.decision_function(X)
        # NEW (Step 2): complement weights describe the OTHER classes,
        # so the best class is the one whose complement fits WORST
        if self.complement:
            return self.classes_[scores.argmin(axis=1)]
        return self.classes_[scores.argmax(axis=1)]

In [20]:
# columns: code, goal, vote
X_toy3 = np.array([[1, 2, 0],   # sports:   "goal goal code"
                   [0, 2, 0],   # sports:   "goal goal"
                   [0, 1, 1],   # sports:   "goal vote"
                   [0, 1, 2],   # politics: "vote vote goal"
                   [2, 0, 1]])  # tech:     "code code vote"
y_toy3 = np.array(["sports", "sports", "sports", "politics", "tech"])

cnb_toy = MyNB(complement=True).fit(X_toy3, y_toy3)
print("classes:", cnb_toy.classes_)
print("weights:\n", cnb_toy.w_.round(3))

X_new3 = np.array([[1, 0, 1]])  # "vote code"
print("scores:", cnb_toy.decision_function(X_new3).round(3))
print("prediction:", cnb_toy.predict(X_new3))

# Two-class check: on the Step 1 toy data, CNB must agree with MNB
print("2-class CNB:", MyNB(complement=True).fit(X_toy, y_toy).predict(X_new),
      "| 2-class MNB:", MyNB(complement=False).fit(X_toy, y_toy).predict(X_new))

classes: ['politics' 'sports' 'tech']
weights:
 [[-1.179 -0.773 -1.466]
 [-1.099 -1.504 -0.811]
 [-1.872 -0.619 -1.179]]
scores: [[-2.645 -1.91  -3.05 ]]
prediction: ['tech']
2-class CNB: ['sports'] | 2-class MNB: ['sports']


In [21]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters

def label(f):
    return reuters.categories(f)[0]

# Keep only documents with exactly one topic
train_ids = [f for f in reuters.fileids()
             if f.startswith("training/") and len(reuters.categories(f)) == 1]
test_ids = [f for f in reuters.fileids()
            if f.startswith("test/") and len(reuters.categories(f)) == 1]

# Keep only topics that appear in both training and test
shared = {label(f) for f in train_ids} & {label(f) for f in test_ids}
train_ids = [f for f in train_ids if label(f) in shared]
test_ids = [f for f in test_ids if label(f) in shared]

yr_train = np.array([label(f) for f in train_ids])
yr_test = np.array([label(f) for f in test_ids])

vectorizer_r = CountVectorizer()
Xr_train = vectorizer_r.fit_transform([reuters.raw(f) for f in train_ids])
Xr_test = vectorizer_r.transform([reuters.raw(f) for f in test_ids])

classes_r, sizes_r = np.unique(yr_train, return_counts=True)
print(f"Reuters: {len(yr_train)} train / {len(yr_test)} test documents, {len(classes_r)} classes")
print(f"Largest class: {classes_r[sizes_r.argmax()]} ({sizes_r.max()} training docs); "
      f"smallest: {sizes_r.min()} training docs")

Reuters: 6560 train / 2570 test documents, 52 classes
Largest class: earn (2840 training docs); smallest: 1 training docs


In [22]:
datasets = {
    "20NG":    (X_train, X_test, y_train, y_test),       # from Step 1
    "Reuters": (Xr_train, Xr_test, yr_train, yr_test),
}
results = []

def evaluate(name, model, dataset):
    X_tr, X_te, y_tr, y_te = datasets[dataset]
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - start
    pred = model.predict(X_te)
    row = {"dataset": dataset, "model": name,
           "accuracy": round(accuracy_score(y_te, pred), 4),
           "macro_f1": round(f1_score(y_te, pred, average="macro", zero_division=0), 4),
           "fit_time_s": round(fit_time, 3)}
    results.append(row)
    print(row)
    return pred

for ds in datasets:
    evaluate("MNB", MyNB(complement=False), ds)
    evaluate("CNB", MyNB(complement=True), ds)

{'dataset': '20NG', 'model': 'MNB', 'accuracy': 0.8621, 'macro_f1': 0.8465, 'fit_time_s': 0.258}
{'dataset': '20NG', 'model': 'CNB', 'accuracy': 0.9034, 'macro_f1': 0.8986, 'fit_time_s': 0.192}
{'dataset': 'Reuters', 'model': 'MNB', 'accuracy': 0.8471, 'macro_f1': 0.2334, 'fit_time_s': 0.061}
{'dataset': 'Reuters', 'model': 'CNB', 'accuracy': 0.8977, 'macro_f1': 0.5285, 'fit_time_s': 0.074}


In [23]:
from sklearn.naive_bayes import ComplementNB

for ds, (X_tr, X_te, y_tr, y_te) in datasets.items():
    # MNB must still match after our edit (a "regression check")
    ours_m = MyNB(complement=False).fit(X_tr, y_tr)
    ref_m = MultinomialNB(alpha=1.0, fit_prior=False).fit(X_tr, y_tr)
    # CNB check: scikit-learn stores the weights with the opposite sign, hence the +
    ours_c = MyNB(complement=True).fit(X_tr, y_tr)
    ref_c = ComplementNB(alpha=1.0, norm=False).fit(X_tr, y_tr)
    print(ds,
          "| MNB same:", np.array_equal(ours_m.predict(X_te), ref_m.predict(X_te)),
          "| CNB same:", np.array_equal(ours_c.predict(X_te), ref_c.predict(X_te)),
          "| CNB max weight diff:", np.abs(ours_c.w_ + ref_c.feature_log_prob_).max())

20NG | MNB same: True | CNB same: True | CNB max weight diff: 0.0
Reuters | MNB same: True | CNB same: True | CNB max weight diff: 0.0


In [24]:
import pandas as pd
from sklearn.metrics import recall_score

pred_m = MyNB(complement=False).fit(Xr_train, yr_train).predict(Xr_test)
pred_c = MyNB(complement=True).fit(Xr_train, yr_train).predict(Xr_test)

table = pd.DataFrame({
    "train_docs": sizes_r,
    "MNB_recall": recall_score(yr_test, pred_m, labels=classes_r, average=None, zero_division=0),
    "CNB_recall": recall_score(yr_test, pred_c, labels=classes_r, average=None, zero_division=0),
}, index=classes_r).sort_values("train_docs")

print(table.round(2).to_string())

small = table["train_docs"] <= table["train_docs"].median()
print("\nMean recall, SMALL half of classes:\n", table[small][["MNB_recall", "CNB_recall"]].mean().round(3))
print("\nMean recall, LARGE half of classes:\n", table[~small][["MNB_recall", "CNB_recall"]].mean().round(3))

                 train_docs  MNB_recall  CNB_recall
platinum                  1        0.00        0.00
jet                       2        0.00        0.00
potato                    2        0.00        0.00
tea                       2        0.00        0.33
nickel                    3        0.00        1.00
dlr                       3        0.00        0.00
cpu                       3        0.00        0.00
fuel                      4        0.00        0.00
lead                      4        0.00        0.00
instal-debt               5        0.00        0.00
heat                      6        0.00        0.50
carcass                   6        0.00        0.00
income                    7        0.00        0.00
zinc                      8        0.00        0.60
strategic-metal           9        0.00        0.00
gas                      10        0.00        0.75
lumber                   10        0.00        0.50
meal-feed                10        0.00        0.00
lei         

#WNCNB

In [25]:
# Table 4, step 6 EXACTLY as printed: divide by the plain sum (no absolute value)
def trap_predict(cnb_model, X):
    w_trap = cnb_model.w_ / cnb_model.w_.sum(axis=1, keepdims=True)
    scores = np.asarray(sparse.csr_matrix(X, dtype=np.float64) @ w_trap.T)
    return cnb_model.classes_[scores.argmin(axis=1)], w_trap

# columns (alphabetical): boston, city, francisco, san
X_sf = np.array([[5, 10, 0, 0],    # Boston doc
                 [5, 10, 0, 0],    # Boston doc
                 [0, 5, 5, 5],     # SF doc
                 [0, 5, 5, 5]])    # SF doc
y_sf = np.array(["boston", "boston", "sf", "sf"])
t = np.array([[5, 0, 3, 3]])       # "boston" x5, "san francisco" x3

pred_toy, w_trap_toy = trap_predict(MyNB(complement=True).fit(X_sf, y_sf), t)
print("Trap on toy:", pred_toy, "| all weights now positive?", (w_trap_toy > 0).all())

pred_trap, _ = trap_predict(MyNB(complement=True).fit(X_train, y_train), X_test)
print("Trap accuracy on 20NG:", round(accuracy_score(y_test, pred_trap), 4))

Trap on toy: ['sf'] | all weights now positive? True
Trap accuracy on 20NG: 0.0194


In [27]:
class MyNB:
    def __init__(self, alpha=1.0, complement=False, weight_norm=False):
        self.alpha = alpha
        self.complement = complement
        self.weight_norm = weight_norm

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_words = X.shape[1]

        counts = np.vstack([
            np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_
        ])

        # Step 2 (Eq. 6): complement counts
        if self.complement:
            counts = counts.sum(axis=0, keepdims=True) - counts

        theta = (counts + self.alpha) / (
            counts.sum(axis=1, keepdims=True) + self.alpha * n_words)
        w = np.log(theta)

        # NEW (Step 3, Eq. 8): divide each class's weights by the sum of their
        # ABSOLUTE values. Weights stay negative, so arg min still works.
        if self.weight_norm:
            w = w / np.abs(w).sum(axis=1, keepdims=True)

        self.w_ = w
        return self

    def decision_function(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64)
        return np.asarray(X @ self.w_.T)

    def predict(self, X):
        scores = self.decision_function(X)
        if self.complement:
            return self.classes_[scores.argmin(axis=1)]
        return self.classes_[scores.argmax(axis=1)]

In [28]:
for name, model in [("MNB", MyNB()),
                    ("MNB + weight norm", MyNB(weight_norm=True)),
                    ("CNB", MyNB(complement=True)),
                    ("WCNB", MyNB(complement=True, weight_norm=True))]:
    model.fit(X_sf, y_sf)
    print(f"{name:18s} scores: {model.decision_function(t).round(3)} -> {model.predict(t)}")

MNB                scores: [[-26.8   -24.403]] -> ['sf']
MNB + weight norm  scores: [[-3.094 -3.531]] -> ['boston']
CNB                scores: [[-24.403 -26.8  ]] -> ['sf']
WCNB               scores: [[-3.531 -3.094]] -> ['boston']


In [29]:
for ds in datasets:
    evaluate("WCNB", MyNB(complement=True, weight_norm=True), ds)

print()
print(pd.DataFrame(results).to_string(index=False))

{'dataset': '20NG', 'model': 'WCNB', 'accuracy': 0.8814, 'macro_f1': 0.8713, 'fit_time_s': 0.153}
{'dataset': 'Reuters', 'model': 'WCNB', 'accuracy': 0.8981, 'macro_f1': 0.5172, 'fit_time_s': 0.078}

dataset model  accuracy  macro_f1  fit_time_s
   20NG   MNB    0.8621    0.8465       0.258
   20NG   CNB    0.9034    0.8986       0.192
Reuters   MNB    0.8471    0.2334       0.061
Reuters   CNB    0.8977    0.5285       0.074
   20NG  WCNB    0.8814    0.8713       0.153
Reuters  WCNB    0.8981    0.5172       0.078


In [30]:
for ds, (X_tr, X_te, y_tr, y_te) in datasets.items():
    ours_w = MyNB(complement=True, weight_norm=True).fit(X_tr, y_tr)
    ref_w = ComplementNB(alpha=1.0, norm=True).fit(X_tr, y_tr)
    ours_c = MyNB(complement=True).fit(X_tr, y_tr)              # regression check
    ref_c = ComplementNB(alpha=1.0, norm=False).fit(X_tr, y_tr)
    print(ds,
          "| WCNB same:", np.array_equal(ours_w.predict(X_te), ref_w.predict(X_te)),
          "| WCNB max weight diff:", np.abs(ours_w.w_ + ref_w.feature_log_prob_).max(),
          "| CNB still same:", np.array_equal(ours_c.predict(X_te), ref_c.predict(X_te)))

20NG | WCNB same: True | WCNB max weight diff: 0.0 | CNB still same: True
Reuters | WCNB same: True | WCNB max weight diff: 0.0 | CNB still same: True


In [31]:
pred_w = MyNB(complement=True, weight_norm=True).fit(Xr_train, yr_train).predict(Xr_test)
table["WCNB_recall"] = recall_score(yr_test, pred_w, labels=table.index,
                                    average=None, zero_division=0)

cols = ["MNB_recall", "CNB_recall", "WCNB_recall"]
small = table["train_docs"] <= table["train_docs"].median()
print("Mean recall, SMALL half:\n", table[small][cols].mean().round(3))
print("\nMean recall, LARGE half:\n", table[~small][cols].mean().round(3))

Mean recall, SMALL half:
 MNB_recall     0.004
CNB_recall     0.320
WCNB_recall    0.285
dtype: float64

Mean recall, LARGE half:
 MNB_recall     0.440
CNB_recall     0.654
WCNB_recall    0.661
dtype: float64


#T-MNB

In [32]:
class MyTransformer:
    """Table 4, steps 1-3: log TF, IDF and length normalisation."""

    def __init__(self, log_tf=True, idf=True, length_norm=True):
        self.log_tf = log_tf
        self.idf = idf
        self.length_norm = length_norm

    def fit(self, X, y=None):
        X = sparse.csr_matrix(X)
        n_docs = X.shape[0]
        # Document frequency: how many training documents contain each word
        df = np.asarray((X > 0).sum(axis=0)).ravel()
        # Section 4.2: idf = log(number of documents / documents containing the word)
        self.idf_ = np.log(n_docs / np.maximum(df, 1))
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64, copy=True)

        # Step 1 (Sec. 4.1): log TF. X.data holds only the NON-ZERO counts.
        if self.log_tf:
            X.data = np.log2(1.0 + X.data)

        # Step 2 (Sec. 4.2): multiply column i by idf_i
        if self.idf:
            X = (X @ sparse.diags(self.idf_)).tocsr()

        # Step 3 (Sec. 4.3): divide row j by its Euclidean length
        if self.length_norm:
            norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
            norms[norms == 0] = 1.0           # avoid dividing an empty document by 0
            X = (sparse.diags(1.0 / norms) @ X).tocsr()
        return X

In [33]:
# columns (alphabetical): goal, match, the
X_tf = np.array([[3, 1, 7],
                 [0, 1, 3],
                 [1, 0, 1]])

step1 = MyTransformer(log_tf=True, idf=False, length_norm=False).fit(X_tf)
step2 = MyTransformer(log_tf=True, idf=True,  length_norm=False).fit(X_tf)
step3 = MyTransformer(log_tf=True, idf=True,  length_norm=True).fit(X_tf)

print("IDF per word:", step2.idf_.round(3))
print("after log TF:\n", step1.transform(X_tf).toarray().round(3))
print("after log TF + IDF:\n", step2.transform(X_tf).toarray().round(3))
print("after all three:\n", step3.transform(X_tf).toarray().round(3))

IDF per word: [0.405 0.405 0.   ]
after log TF:
 [[2. 1. 3.]
 [0. 1. 2.]
 [1. 0. 1.]]
after log TF + IDF:
 [[0.811 0.405 0.   ]
 [0.    0.405 0.   ]
 [0.405 0.    0.   ]]
after all three:
 [[0.894 0.447 0.   ]
 [0.    1.    0.   ]
 [1.    0.    0.   ]]


In [34]:
class MyNB:
    def __init__(self, alpha=1.0, complement=False, weight_norm=False,
                 log_tf=False, idf=False, length_norm=False):
        self.alpha = alpha
        self.complement = complement
        self.weight_norm = weight_norm
        self.log_tf = log_tf
        self.idf = idf
        self.length_norm = length_norm

    def fit(self, X, y):
        y = np.asarray(y)
        # NEW (Step 4): learn IDF from the TRAINING documents, then transform them
        self.transformer_ = MyTransformer(self.log_tf, self.idf, self.length_norm).fit(X)
        X = self.transformer_.transform(X)

        self.classes_ = np.unique(y)
        n_words = X.shape[1]
        counts = np.vstack([
            np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_
        ])
        if self.complement:                                          # Step 2
            counts = counts.sum(axis=0, keepdims=True) - counts
        theta = (counts + self.alpha) / (
            counts.sum(axis=1, keepdims=True) + self.alpha * n_words)
        w = np.log(theta)
        if self.weight_norm:                                         # Step 3
            w = w / np.abs(w).sum(axis=1, keepdims=True)
        self.w_ = w
        return self

    def decision_function(self, X):
        # NEW (Step 4): test documents get the SAME transforms, using the training IDF
        X = self.transformer_.transform(X)
        return np.asarray(X @ self.w_.T)

    def predict(self, X):
        scores = self.decision_function(X)
        if self.complement:
            return self.classes_[scores.argmin(axis=1)]
        return self.classes_[scores.argmax(axis=1)]

In [35]:
tr = MyTransformer().fit(X_train)
Xt = tr.transform(X_train)
n_classes = len(np.unique(y_train))
print("Average class total, raw counts:        ", round(X_train.sum() / n_classes))
print("Average class total, after transforms:  ", round(Xt.sum() / n_classes))
print("Smoothing 'pretend counts' (alpha x vocabulary):", X_train.shape[1])

Average class total, raw counts:         217789
Average class total, after transforms:   6986
Smoothing 'pretend counts' (alpha x vocabulary): 152615


In [36]:
transform_results = []

def evaluate(name, model, dataset, store=None):
    X_tr, X_te, y_tr, y_te = datasets[dataset]
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - start
    pred = model.predict(X_te)
    row = {"dataset": dataset, "model": name,
           "accuracy": round(accuracy_score(y_te, pred), 4),
           "macro_f1": round(f1_score(y_te, pred, average="macro", zero_division=0), 4),
           "fit_time_s": round(fit_time, 3)}
    (results if store is None else store).append(row)
    print(row)
    return pred

single_transforms = {
    "MNB + log TF only":      dict(log_tf=True),
    "MNB + IDF only":         dict(idf=True),
    "MNB + length norm only": dict(length_norm=True),
}

for ds in datasets:
    # Which transform matters most on its own? (paper, Sec. 4.4: length > log > IDF)
    for name, opts in single_transforms.items():
        evaluate(name, MyNB(**opts), ds, store=transform_results)
    # Does lighter smoothing help once the data is transformed?
    for a in [0.1, 0.01]:
        evaluate(f"T-MNB alpha={a}",
                 MyNB(alpha=a, log_tf=True, idf=True, length_norm=True),
                 ds, store=transform_results)
    # The paper's setting (alpha = 1) goes into the main results table
    evaluate("T-MNB", MyNB(log_tf=True, idf=True, length_norm=True), ds)

print("\nTransform experiments:")
print(pd.DataFrame(transform_results).to_string(index=False))
print("\nMain results so far:")
print(pd.DataFrame(results).to_string(index=False))

{'dataset': '20NG', 'model': 'MNB + log TF only', 'accuracy': 0.8857, 'macro_f1': 0.8778, 'fit_time_s': 0.176}
{'dataset': '20NG', 'model': 'MNB + IDF only', 'accuracy': 0.9, 'macro_f1': 0.8935, 'fit_time_s': 0.278}
{'dataset': '20NG', 'model': 'MNB + length norm only', 'accuracy': 0.8003, 'macro_f1': 0.7665, 'fit_time_s': 0.274}
{'dataset': '20NG', 'model': 'T-MNB alpha=0.1', 'accuracy': 0.9231, 'macro_f1': 0.9195, 'fit_time_s': 0.485}
{'dataset': '20NG', 'model': 'T-MNB alpha=0.01', 'accuracy': 0.9236, 'macro_f1': 0.9216, 'fit_time_s': 0.922}
{'dataset': '20NG', 'model': 'T-MNB', 'accuracy': 0.8862, 'macro_f1': 0.8751, 'fit_time_s': 0.692}
{'dataset': 'Reuters', 'model': 'MNB + log TF only', 'accuracy': 0.8327, 'macro_f1': 0.2044, 'fit_time_s': 0.23}
{'dataset': 'Reuters', 'model': 'MNB + IDF only', 'accuracy': 0.8911, 'macro_f1': 0.5707, 'fit_time_s': 0.139}
{'dataset': 'Reuters', 'model': 'MNB + length norm only', 'accuracy': 0.6977, 'macro_f1': 0.0527, 'fit_time_s': 0.118}
{'datas

In [37]:
tr = MyTransformer().fit(X_train)

# 1. Wiring: our T-MNB must equal scikit-learn's MNB trained on our transformed data
ours = MyNB(log_tf=True, idf=True, length_norm=True).fit(X_train, y_train)
ref = MultinomialNB(alpha=1.0, fit_prior=False).fit(tr.transform(X_train), y_train)
print("T-MNB same as scikit-learn on transformed data:",
      np.array_equal(ours.predict(X_test), ref.predict(tr.transform(X_test))),
      "| max weight diff:", np.abs(ours.w_ - ref.feature_log_prob_).max())

# 2. Every non-empty transformed test document has length exactly 1
Xt_test = tr.transform(X_test)
norms = np.sqrt(np.asarray(Xt_test.multiply(Xt_test).sum(axis=1)).ravel())
print("Non-empty test docs all unit length:", np.allclose(norms[norms > 0], 1.0),
      "| empty test docs:", int((norms == 0).sum()))

# 3. Regression check: plain MNB is unchanged by this step's edits
print("Plain MNB still matches scikit-learn:",
      np.array_equal(MyNB().fit(X_train, y_train).predict(X_test),
                     MultinomialNB(alpha=1.0, fit_prior=False).fit(X_train, y_train).predict(X_test)))

T-MNB same as scikit-learn on transformed data: True | max weight diff: 1.7763568394002505e-15
Non-empty test docs all unit length: True | empty test docs: 0
Plain MNB still matches scikit-learn: True


#TCNB

In [38]:
for ds, (X_tr, X_te, y_tr, y_te) in datasets.items():
    Xt = MyTransformer().fit(X_tr).transform(X_tr)
    classes = np.unique(y_tr)
    own = np.array([Xt[y_tr == c].sum() for c in classes])
    comp = Xt.sum() - own
    print(f"{ds}: own-class totals {own.min():.0f} to {own.max():.0f} | "
          f"complement totals {comp.min():.0f} to {comp.max():.0f} | "
          f"pretend counts at alpha=1: {X_tr.shape[1]}")

20NG: own-class totals 5201 to 8763 | complement totals 130966 to 134527 | pretend counts at alpha=1: 152615
Reuters: own-class totals 7 to 15789 | complement totals 26445 to 42227 | pretend counts at alpha=1: 23774


In [39]:
start = len(transform_results)

for ds in datasets:
    evaluate("MNB a=0.01, no transforms", MyNB(alpha=0.01), ds, store=transform_results)
    for name, opts in single_transforms.items():
        evaluate(name.replace("MNB", "MNB a=0.01"), MyNB(alpha=0.01, **opts),
                 ds, store=transform_results)
        evaluate(name.replace("MNB", "CNB"), MyNB(complement=True, **opts),
                 ds, store=transform_results)

print(pd.DataFrame(transform_results[start:]).to_string(index=False))

{'dataset': '20NG', 'model': 'MNB a=0.01, no transforms', 'accuracy': 0.9008, 'macro_f1': 0.8944, 'fit_time_s': 0.107}
{'dataset': '20NG', 'model': 'MNB a=0.01 + log TF only', 'accuracy': 0.9159, 'macro_f1': 0.9138, 'fit_time_s': 0.219}
{'dataset': '20NG', 'model': 'CNB + log TF only', 'accuracy': 0.9191, 'macro_f1': 0.9143, 'fit_time_s': 0.271}
{'dataset': '20NG', 'model': 'MNB a=0.01 + IDF only', 'accuracy': 0.9053, 'macro_f1': 0.9029, 'fit_time_s': 0.305}
{'dataset': '20NG', 'model': 'CNB + IDF only', 'accuracy': 0.8968, 'macro_f1': 0.8934, 'fit_time_s': 0.266}
{'dataset': '20NG', 'model': 'MNB a=0.01 + length norm only', 'accuracy': 0.9183, 'macro_f1': 0.9147, 'fit_time_s': 0.224}
{'dataset': '20NG', 'model': 'CNB + length norm only', 'accuracy': 0.8897, 'macro_f1': 0.8787, 'fit_time_s': 0.255}
{'dataset': 'Reuters', 'model': 'MNB a=0.01, no transforms', 'accuracy': 0.8965, 'macro_f1': 0.58, 'fit_time_s': 0.067}
{'dataset': 'Reuters', 'model': 'MNB a=0.01 + log TF only', 'accuracy'

In [40]:
start = len(transform_results)

for ds in datasets:
    for a in [0.1, 0.01]:
        evaluate(f"TCNB alpha={a}",
                 MyNB(alpha=a, complement=True, log_tf=True, idf=True, length_norm=True),
                 ds, store=transform_results)
    # The paper's setting (alpha = 1) goes into the main results table
    evaluate("TCNB", MyNB(complement=True, log_tf=True, idf=True, length_norm=True), ds)

print("\nTCNB smoothing sweep:")
print(pd.DataFrame(transform_results[start:]).to_string(index=False))
print("\nMain results so far:")
print(pd.DataFrame(results).to_string(index=False))

{'dataset': '20NG', 'model': 'TCNB alpha=0.1', 'accuracy': 0.9183, 'macro_f1': 0.9151, 'fit_time_s': 0.841}
{'dataset': '20NG', 'model': 'TCNB alpha=0.01', 'accuracy': 0.9074, 'macro_f1': 0.9043, 'fit_time_s': 0.985}
{'dataset': '20NG', 'model': 'TCNB', 'accuracy': 0.9167, 'macro_f1': 0.9111, 'fit_time_s': 0.482}
{'dataset': 'Reuters', 'model': 'TCNB alpha=0.1', 'accuracy': 0.8774, 'macro_f1': 0.5378, 'fit_time_s': 0.147}
{'dataset': 'Reuters', 'model': 'TCNB alpha=0.01', 'accuracy': 0.8537, 'macro_f1': 0.5172, 'fit_time_s': 0.111}
{'dataset': 'Reuters', 'model': 'TCNB', 'accuracy': 0.8689, 'macro_f1': 0.4277, 'fit_time_s': 0.115}

TCNB smoothing sweep:
dataset           model  accuracy  macro_f1  fit_time_s
   20NG  TCNB alpha=0.1    0.9183    0.9151       0.841
   20NG TCNB alpha=0.01    0.9074    0.9043       0.985
Reuters  TCNB alpha=0.1    0.8774    0.5378       0.147
Reuters TCNB alpha=0.01    0.8537    0.5172       0.111

Main results so far:
dataset model  accuracy  macro_f1  f

In [41]:
tr = MyTransformer().fit(X_train)
ours = MyNB(complement=True, log_tf=True, idf=True, length_norm=True).fit(X_train, y_train)
ref = ComplementNB(alpha=1.0, norm=False).fit(tr.transform(X_train), y_train)
print("TCNB same as scikit-learn on transformed data:",
      np.array_equal(ours.predict(X_test), ref.predict(tr.transform(X_test))),
      "| max weight diff:", np.abs(ours.w_ + ref.feature_log_prob_).max())

TCNB same as scikit-learn on transformed data: True | max weight diff: 1.7763568394002505e-15


In [42]:
def add_recall(column, model):
    pred = model.fit(Xr_train, yr_train).predict(Xr_test)
    table[column] = recall_score(yr_test, pred, labels=table.index,
                                 average=None, zero_division=0)

add_recall("TMNB_a001", MyNB(alpha=0.01, log_tf=True, idf=True, length_norm=True))
add_recall("TCNB", MyNB(complement=True, log_tf=True, idf=True, length_norm=True))
add_recall("TCNB_a001", MyNB(alpha=0.01, complement=True, log_tf=True, idf=True, length_norm=True))

cols = ["MNB_recall", "CNB_recall", "TMNB_a001", "TCNB", "TCNB_a001"]
small = table["train_docs"] <= table["train_docs"].median()
print("Mean recall, SMALL half:\n", table[small][cols].mean().round(3))
print("\nMean recall, LARGE half:\n", table[~small][cols].mean().round(3))

Mean recall, SMALL half:
 MNB_recall    0.004
CNB_recall    0.320
TMNB_a001     0.399
TCNB          0.192
TCNB_a001     0.328
dtype: float64

Mean recall, LARGE half:
 MNB_recall    0.440
CNB_recall    0.654
TMNB_a001     0.795
TCNB          0.561
TCNB_a001     0.626
dtype: float64


#TWCNB

In [43]:
TWCNB_OPTS = dict(complement=True, weight_norm=True, log_tf=True, idf=True, length_norm=True)
start = len(transform_results)

for ds in datasets:
    for a in [0.1, 0.01]:
        evaluate(f"TWCNB alpha={a}", MyNB(alpha=a, **TWCNB_OPTS), ds, store=transform_results)
    # The paper's setting (alpha = 1) goes into the main results table
    evaluate("TWCNB", MyNB(**TWCNB_OPTS), ds)

print("\nTWCNB smoothing sweep:")
print(pd.DataFrame(transform_results[start:]).to_string(index=False))

{'dataset': '20NG', 'model': 'TWCNB alpha=0.1', 'accuracy': 0.9178, 'macro_f1': 0.9146, 'fit_time_s': 0.489}
{'dataset': '20NG', 'model': 'TWCNB alpha=0.01', 'accuracy': 0.9042, 'macro_f1': 0.9009, 'fit_time_s': 0.716}
{'dataset': '20NG', 'model': 'TWCNB', 'accuracy': 0.9146, 'macro_f1': 0.9086, 'fit_time_s': 0.986}
{'dataset': 'Reuters', 'model': 'TWCNB alpha=0.1', 'accuracy': 0.8837, 'macro_f1': 0.5667, 'fit_time_s': 0.197}
{'dataset': 'Reuters', 'model': 'TWCNB alpha=0.01', 'accuracy': 0.8109, 'macro_f1': 0.5084, 'fit_time_s': 0.232}
{'dataset': 'Reuters', 'model': 'TWCNB', 'accuracy': 0.835, 'macro_f1': 0.3255, 'fit_time_s': 0.231}

TWCNB smoothing sweep:
dataset            model  accuracy  macro_f1  fit_time_s
   20NG  TWCNB alpha=0.1    0.9178    0.9146       0.489
   20NG TWCNB alpha=0.01    0.9042    0.9009       0.716
Reuters  TWCNB alpha=0.1    0.8837    0.5667       0.197
Reuters TWCNB alpha=0.01    0.8109    0.5084       0.232


In [44]:
tr = MyTransformer().fit(X_train)
Xt_tr, Xt_te = tr.transform(X_train), tr.transform(X_test)

for name, opts, norm in [("TCNB", dict(complement=True, log_tf=True, idf=True, length_norm=True), False),
                         ("TWCNB", TWCNB_OPTS, True)]:
    ours = MyNB(**opts).fit(X_train, y_train)
    ref = ComplementNB(alpha=1.0, norm=norm).fit(Xt_tr, y_train)
    print(f"{name} same as scikit-learn on transformed data:",
          np.array_equal(ours.predict(X_test), ref.predict(Xt_te)),
          "| max weight diff:", np.abs(ours.w_ + ref.feature_log_prob_).max())

TCNB same as scikit-learn on transformed data: True | max weight diff: 1.7763568394002505e-15
TWCNB same as scikit-learn on transformed data: True | max weight diff: 1.6940658945086007e-21


In [45]:
from sklearn.svm import LinearSVC

class TransformedSVM:
    """The paper's baseline: linear SVM, one-vs-rest, C = 10, same transformed features."""
    def __init__(self, C=10.0):
        self.C = C

    def fit(self, X, y):
        self.transformer_ = MyTransformer().fit(X)
        self.svm_ = LinearSVC(C=self.C, dual=True, max_iter=10000)
        self.svm_.fit(self.transformer_.transform(X), y)
        return self

    def predict(self, X):
        return self.svm_.predict(self.transformer_.transform(X))

for ds in datasets:
    evaluate("LinearSVC", TransformedSVM(C=10.0), ds)

{'dataset': '20NG', 'model': 'LinearSVC', 'accuracy': 0.939, 'macro_f1': 0.9378, 'fit_time_s': 29.954}
{'dataset': 'Reuters', 'model': 'LinearSVC', 'accuracy': 0.9471, 'macro_f1': 0.7682, 'fit_time_s': 4.613}


In [46]:
df = pd.DataFrame(results)
for ds in datasets:
    nb = df[(df.dataset == ds) & (df.model == "TWCNB")].iloc[0]
    svm = df[(df.dataset == ds) & (df.model == "LinearSVC")].iloc[0]
    print(f"{ds}: accuracy TWCNB {nb.accuracy} vs SVM {svm.accuracy} | "
          f"macro-F1 TWCNB {nb.macro_f1} vs SVM {svm.macro_f1} | "
          f"SVM took {svm.fit_time_s / nb.fit_time_s:.0f}x longer to train")

print("\nMain results table (all at the paper's settings):")
print(df.to_string(index=False))

20NG: accuracy TWCNB 0.9146 vs SVM 0.939 | macro-F1 TWCNB 0.9086 vs SVM 0.9378 | SVM took 30x longer to train
Reuters: accuracy TWCNB 0.835 vs SVM 0.9471 | macro-F1 TWCNB 0.3255 vs SVM 0.7682 | SVM took 20x longer to train

Main results table (all at the paper's settings):
dataset     model  accuracy  macro_f1  fit_time_s
   20NG       MNB    0.8621    0.8465       0.258
   20NG       CNB    0.9034    0.8986       0.192
Reuters       MNB    0.8471    0.2334       0.061
Reuters       CNB    0.8977    0.5285       0.074
   20NG      WCNB    0.8814    0.8713       0.153
Reuters      WCNB    0.8981    0.5172       0.078
   20NG     T-MNB    0.8862    0.8751       0.692
Reuters     T-MNB    0.7829    0.1494       0.200
   20NG      TCNB    0.9167    0.9111       0.482
Reuters      TCNB    0.8689    0.4277       0.115
   20NG     TWCNB    0.9146    0.9086       0.986
Reuters     TWCNB    0.8350    0.3255       0.231
   20NG LinearSVC    0.9390    0.9378      29.954
Reuters LinearSVC    0.947

In [47]:
add_recall("TWCNB", MyNB(**TWCNB_OPTS))
add_recall("SVM", TransformedSVM(C=10.0))

cols = ["MNB_recall", "CNB_recall", "TMNB_a001", "TCNB", "TWCNB", "SVM"]
small = table["train_docs"] <= table["train_docs"].median()
print("Mean recall, SMALL half:\n", table[small][cols].mean().round(3))
print("\nMean recall, LARGE half:\n", table[~small][cols].mean().round(3))

Mean recall, SMALL half:
 MNB_recall    0.004
CNB_recall    0.320
TMNB_a001     0.399
TCNB          0.192
TWCNB         0.112
SVM           0.609
dtype: float64

Mean recall, LARGE half:
 MNB_recall    0.440
CNB_recall    0.654
TMNB_a001     0.795
TCNB          0.561
TWCNB         0.435
SVM           0.888
dtype: float64


#final comparison

In [48]:
class MyTransformer:
    """Table 4, steps 1-3: log TF, IDF and length normalisation."""

    def __init__(self, log_tf=True, idf=True, length_norm=True):
        self.log_tf = log_tf
        self.idf = idf
        self.length_norm = length_norm

    def fit(self, X, y=None):
        X = sparse.csr_matrix(X)
        n_docs = X.shape[0]
        df = np.asarray((X > 0).sum(axis=0)).ravel()
        # Words never seen in these training documents get IDF 0 (no information)
        self.idf_ = np.where(df > 0, np.log(n_docs / np.maximum(df, 1)), 0.0)
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64, copy=True)
        if self.log_tf:
            X.data = np.log2(1.0 + X.data)
        if self.idf:
            X = (X @ sparse.diags(self.idf_)).tocsr()
        if self.length_norm:
            norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
            norms[norms == 0] = 1.0
            X = (sparse.diags(1.0 / norms) @ X).tocsr()
        return X

In [49]:
ALPHAS = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003]
C_GRID = [0.1, 1.0, 10.0]
VARIANTS = {
    "MNB":   dict(),
    "CNB":   dict(complement=True),
    "WCNB":  dict(complement=True, weight_norm=True),
    "T-MNB": dict(log_tf=True, idf=True, length_norm=True),
    "TCNB":  dict(complement=True, log_tf=True, idf=True, length_norm=True),
    "TWCNB": dict(complement=True, weight_norm=True, log_tf=True, idf=True, length_norm=True),
}

def validation_split(X, y, seed):
    """Carve 20% of the TRAINING data out as a validation set. The test set is never touched."""
    _, counts = np.unique(y, return_counts=True)
    strat = y if counts.min() >= 2 else None    # Reuters has 1-document classes: can't stratify
    return train_test_split(X, y, test_size=0.2, stratify=strat, random_state=seed)

def pick_best(make_model, grid, X_tr, y_tr, seed):
    """Return the grid value with the best macro-F1 on the validation set."""
    X_fit, X_val, y_fit, y_val = validation_split(X_tr, y_tr, seed)
    scores = {}
    for value in grid:
        pred = make_model(value).fit(X_fit, y_fit).predict(X_val)
        scores[value] = f1_score(y_val, pred, average="macro", zero_division=0)
    return max(scores, key=scores.get)

def score(model, X_tr, y_tr, X_te, y_te):
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - start
    pred = model.predict(X_te)
    return dict(accuracy=accuracy_score(y_te, pred),
                macro_f1=f1_score(y_te, pred, average="macro", zero_division=0),
                fit_time_s=fit_time)

In [50]:
def run_split(X_tr, X_te, y_tr, y_te, setting, split):
    rows = []
    for name, opts in VARIANTS.items():
        # Protocol 1: the paper's settings (alpha = 1)
        rows.append(dict(setting=setting, split=split, protocol="paper", model=name, param=1.0,
                         **score(MyNB(alpha=1.0, **opts), X_tr, y_tr, X_te, y_te)))
        # Protocol 2: alpha chosen on validation data carved from the training set
        a = pick_best(lambda v: MyNB(alpha=v, **opts), ALPHAS, X_tr, y_tr, split)
        rows.append(dict(setting=setting, split=split, protocol="tuned", model=name, param=a,
                         **score(MyNB(alpha=a, **opts), X_tr, y_tr, X_te, y_te)))

    # SVM: the paper's C = 10, then C chosen on validation data
    paper_svm = score(TransformedSVM(C=10.0), X_tr, y_tr, X_te, y_te)
    rows.append(dict(setting=setting, split=split, protocol="paper", model="LinearSVC",
                     param=10.0, **paper_svm))
    c = pick_best(lambda v: TransformedSVM(C=v), C_GRID, X_tr, y_tr, split)
    tuned_svm = paper_svm if c == 10.0 else score(TransformedSVM(C=c), X_tr, y_tr, X_te, y_te)
    rows.append(dict(setting=setting, split=split, protocol="tuned", model="LinearSVC",
                     param=c, **tuned_svm))
    return rows

In [52]:
all_rows = []

# 1. 20 Newsgroups, the paper's protocol: 10 random stratified 80/20 splits, headers kept
N_SPLITS = 10            # set this to 2 first for a quick trial run
for split in range(N_SPLITS):
    tr_text, te_text, ytr, yte = train_test_split(
        data.data, data.target, test_size=0.2, stratify=data.target, random_state=split)
    vec = CountVectorizer()
    Xtr, Xte = vec.fit_transform(tr_text), vec.transform(te_text)
    t0 = time.perf_counter()
    all_rows += run_split(Xtr, Xte, ytr, yte, "20NG random", split)
    print(f"20NG random: split {split + 1}/{N_SPLITS} done in {time.perf_counter() - t0:.0f}s")

# 2. Reuters single-label, fixed ModApte split
all_rows += run_split(Xr_train, Xr_test, yr_train, yr_test, "Reuters", 0)
print("Reuters done")

# 3. 20 Newsgroups, chronological split, headers/signatures/quotes removed
remove = ("headers", "footers", "quotes")
bd_train = fetch_20newsgroups(subset="train", remove=remove)
bd_test = fetch_20newsgroups(subset="test", remove=remove)
vec = CountVectorizer()
Xb_tr, Xb_te = vec.fit_transform(bd_train.data), vec.transform(bd_test.data)
all_rows += run_split(Xb_tr, Xb_te, bd_train.target, bd_test.target, "20NG bydate", 0)
print("20NG bydate done")

final = pd.DataFrame(all_rows)
final.to_csv("final_results.csv", index=False)

20NG random: split 1/10 done in 56s
20NG random: split 2/10 done in 49s
20NG random: split 3/10 done in 50s
20NG random: split 4/10 done in 56s
20NG random: split 5/10 done in 55s
20NG random: split 6/10 done in 48s
20NG random: split 7/10 done in 54s
20NG random: split 8/10 done in 50s
20NG random: split 9/10 done in 56s
20NG random: split 10/10 done in 58s
Reuters done
20NG bydate done


In [53]:
order = list(VARIANTS) + ["LinearSVC"]
summary = (final.groupby(["setting", "protocol", "model"])[["accuracy", "macro_f1", "fit_time_s"]]
                .agg(["mean", "std"]).round(4))

for setting in ["20NG random", "Reuters", "20NG bydate"]:
    for protocol in ["paper", "tuned"]:
        print(f"\n=== {setting} | {protocol} ===")
        print(summary.loc[(setting, protocol)].reindex(order).to_string())

print("\nChosen alpha (Naive Bayes) or C (SVM), with how often each was picked:")
tuned = final[final.protocol == "tuned"]
print(tuned.groupby(["setting", "model"])["param"]
           .apply(lambda s: ", ".join(f"{v:g} ({n})" for v, n in s.value_counts().items()))
           .to_string())


=== 20NG random | paper ===
          accuracy         macro_f1         fit_time_s        
              mean     std     mean     std       mean     std
model                                                         
MNB         0.8537  0.0076   0.8375  0.0083     0.2268  0.0304
CNB         0.9009  0.0065   0.8955  0.0070     0.1425  0.0289
WCNB        0.8785  0.0055   0.8672  0.0063     0.1468  0.0227
T-MNB       0.8834  0.0064   0.8716  0.0068     0.3173  0.0450
TCNB        0.9126  0.0065   0.9066  0.0070     0.3379  0.0650
TWCNB       0.9101  0.0067   0.9036  0.0071     0.3544  0.0597
LinearSVC   0.9387  0.0051   0.9377  0.0052    17.5564  1.5604

=== 20NG random | tuned ===
          accuracy         macro_f1         fit_time_s        
              mean     std     mean     std       mean     std
model                                                         
MNB         0.8918  0.0072   0.8872  0.0075     0.1268  0.0200
CNB         0.9034  0.0079   0.8988  0.0087     0.1324  0.01